# Dual-Input End-to-End Neural Network Training Pipeline

This notebook trains **3 Dual-Input End-to-End Neural Networks** (`Dual-NN Hypotension`, `Dual-NN Hypoxia`, `Dual-NN Tachycardia`) that fuse 1D CNN time-series telemetry (600 seconds $\times$ 19 vital features) with patient clinical metadata (65 features).

### Dual-Input Architecture:
1. **Time-Series Branch**: `Input(shape=(600, 19))` $\rightarrow$ 1D Conv $\rightarrow$ BatchNorm $\rightarrow$ MaxPool $\rightarrow$ 1D Conv $\rightarrow$ GlobalAveragePooling1D.
2. **Clinical Metadata Branch**: `Input(shape=(65,))` $\rightarrow$ Dense $\rightarrow$ Batch Normalization.
3. **Feature Fusion Layer**: Concatenate representations $\rightarrow$ Dense $\rightarrow$ Dropout $\rightarrow$ Sigmoid Risk Probability.

### Target Outputs:
- **`Future_Hypotension`**: MAP $< 65\text{ mmHg}$
- **`Future_Hypoxia`**: $\text{SpO}_2 < 90\%$
- **`Future_Tachycardia`**: HR $> 100\text{ bpm}$

## 1. Environment & Setup

In [ ]:
import os
import glob
import json
import re
import numpy as np
import pandas as pd

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns

import tensorflow as tf
from tensorflow import keras
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, f1_score, precision_score, recall_score, roc_auc_score

MODEL_OUTPUT_DIR = 'models/dual_input_nn'
os.makedirs(MODEL_OUTPUT_DIR, exist_ok=True)
print(f'[Init] Output directory initialized: {MODEL_OUTPUT_DIR}')

## 2. Dual-Input Model Architecture Builder

In [ ]:
def build_dual_input_nn(seq_shape=(600, 19), meta_shape=(65,)):
    # 1. Time-Series Telemetry Branch
    input_seq = keras.Input(shape=seq_shape, name='seq_input')
    x_seq = keras.layers.Conv1D(filters=32, kernel_size=5, activation='relu', padding='same')(input_seq)
    x_seq = keras.layers.BatchNormalization()(x_seq)
    x_seq = keras.layers.MaxPooling1D(pool_size=2)(x_seq)
    
    x_seq = keras.layers.Conv1D(filters=64, kernel_size=3, activation='relu', padding='same')(x_seq)
    x_seq = keras.layers.BatchNormalization()(x_seq)
    x_seq = keras.layers.GlobalAveragePooling1D()(x_seq)
    
    # 2. Clinical Metadata Branch
    input_meta = keras.Input(shape=meta_shape, name='meta_input')
    x_meta = keras.layers.Dense(32, activation='relu')(input_meta)
    x_meta = keras.layers.BatchNormalization()(x_meta)
    
    # 3. Concatenation & Fusion
    combined = keras.layers.concatenate([x_seq, x_meta])
    x = keras.layers.Dense(32, activation='relu')(combined)
    x = keras.layers.Dropout(0.3)(x)
    output = keras.layers.Dense(1, activation='sigmoid', name='risk_output')(x)
    
    model = keras.Model(inputs=[input_seq, input_meta], outputs=output)
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=0.001),
        loss='binary_crossentropy',
        metrics=[keras.metrics.AUC(name='auc'), keras.metrics.AUC(curve='PR', name='pr_auc')]
    )
    return model

model = build_dual_input_nn()
model.summary()